# Bigram vs LSTM vs Small Transformer — Next-Character Prediction

**Applied NLP — Midterm Project** · Aashay Gokhale

All three models solve the same task on Tiny Shakespeare: given the preceding
characters, predict the next one. The code lives in plain Python files —
`data.py`, `models.py`, `train.py` — and this notebook runs them in order and
shows the results. Explanations of each model are in the README.

**To run on Colab:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all.

In [1]:
import os, sys
if not os.path.exists("train.py"):           # bare Colab session: fetch the repo
    !git clone -q https://github.com/Aashay1-ops/nlp-midterm.git
    %cd nlp-midterm
sys.path.insert(0, os.getcwd())

# QUICK = True runs a few steps per model to check the notebook end to end.
# The committed results come from QUICK = False.
QUICK = False

/content/nlp-midterm


In [2]:
import math
import torch
import matplotlib.pyplot as plt

import data
from data import decode, device, stoi, itos, vocab_size
from models import bigram_count_loss, count_parameters
from train import CONFIG, DISPLAY, generate, run, save_results, plot_curves

cfg = dict(CONFIG)
if QUICK:
    cfg.update(max_steps=60, warmup_steps=6, eval_interval=30)

print("device:", device, f"({torch.cuda.get_device_name(0)})" if device == "cuda" else "")
print("torch:", torch.__version__)
print({k: cfg[k] for k in ("block_size", "batch_size", "max_steps", "lr")})

device: cuda (Tesla T4)
torch: 2.11.0+cu130
{'block_size': 256, 'batch_size': 64, 'max_steps': 5000, 'lr': {'bigram': 0.1, 'lstm': 0.002, 'transformer': 0.002}}


## 1. Dataset and setup

Tiny Shakespeare, split contiguously: the first 90% for training, the last 10%
for validation. A random split would leak, because neighbouring training
windows overlap almost entirely.

In [3]:
base = data.baselines()
counts = bigram_count_loss(data.train_data, data.val_data, vocab_size)

print(f"characters:       {len(data.text):,}")
print(f"vocabulary:       {vocab_size}  {''.join(data.chars)!r}")
print(f"train / val:      {len(data.train_data):,} / {len(data.val_data):,}")
print(f"val chars unseen in train: {len(set(data.val_data.tolist()) - set(data.train_data.tolist()))}")
print()
print(f"uniform-guess loss  ln({vocab_size}) = {base['uniform']:.4f} nats/char")
print(f"unigram loss (frequencies only) = {base['unigram']:.4f} nats/char")

characters:       1,115,394
vocabulary:       65  "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
train / val:      1,003,854 / 111,540
val chars unseen in train: 0

uniform-guess loss  ln(65) = 4.1744 nats/char
unigram loss (frequencies only) = 3.3473 nats/char


The two baselines say what a loss number means. A model that cannot beat the
unigram loss is not using context at all.

## 2. Model 1 — Bigram

One 65×65 table: row *i* is the logits for the character after character *i*.
It is trained by the same loop as the other two models, so its loss and time
are measured the same way.

### 2.1 How sparse is the table?

In [4]:
print(f"possible pairs:              {vocab_size**2:,}")
print(f"pairs never seen in training: {counts['unseen_pairs']:,}")
print(f"val positions using an unseen pair: {counts['val_positions_with_unseen_pair']}")
print(f"closed-form add-1 bigram val loss:  {counts['val_loss']:.4f}  (best a bigram can do)")

possible pairs:              4,225
pairs never seen in training: 2,845
val positions using an unseen pair: 187
closed-form add-1 bigram val loss:  2.4819  (best a bigram can do)


### 2.2 Training

In [5]:
results, models = {}, {}
models["bigram"], results["bigram"] = run("bigram", cfg)


== Bigram (4,225 parameters) on cuda ==
  step     0 | train 4.1744 | val 4.1744 |    0.0s
  step   250 | train 2.4682 | val 2.4952 |    0.6s
  step   500 | train 2.4601 | val 2.4894 |    0.9s
  step   750 | train 2.4584 | val 2.4893 |    1.3s
  step  1000 | train 2.4582 | val 2.4939 |    1.6s
  step  1250 | train 2.4578 | val 2.4908 |    2.0s
  step  1500 | train 2.4580 | val 2.4902 |    2.4s
  step  1750 | train 2.4572 | val 2.4888 |    2.8s
  step  2000 | train 2.4572 | val 2.4915 |    3.2s
  step  2250 | train 2.4566 | val 2.4899 |    3.6s
  step  2500 | train 2.4564 | val 2.4905 |    3.9s
  step  2750 | train 2.4563 | val 2.4913 |    4.3s
  step  3000 | train 2.4557 | val 2.4889 |    4.6s
  step  3250 | train 2.4556 | val 2.4910 |    5.0s
  step  3500 | train 2.4556 | val 2.4905 |    5.3s
  step  3750 | train 2.4550 | val 2.4885 |    5.6s
  step  4000 | train 2.4549 | val 2.4871 |    6.0s
  step  4250 | train 2.4545 | val 2.4886 |    6.3s
  step  4500 | train 2.4546 | val 2.4882 

### 2.3 What the table learned

The whole bigram model is this table, so it can be read directly.

In [6]:
probs = torch.softmax(models["bigram"].table.weight.detach().cpu(), dim=-1)

def top_next(ch, k=5):
    p, i = probs[stoi[ch]].topk(k)
    return "  ".join(f"{itos[j]!r}:{q:.2f}" for q, j in zip(p.tolist(), i.tolist()))

for ch in ["q", "t", "h", " ", "\n", ":"]:
    print(f"after {ch!r:5}-> {top_next(ch)}")

after 'q'  -> 'u':1.00  'd':0.00  'e':0.00  'f':0.00  'c':0.00
after 't'  -> 'h':0.34  ' ':0.24  'o':0.09  'e':0.07  'i':0.04
after 'h'  -> 'e':0.35  'a':0.19  'i':0.13  'o':0.10  ' ':0.10
after ' '  -> 't':0.14  'a':0.08  'h':0.07  's':0.07  'w':0.06
after '\n' -> '\n':0.18  'T':0.10  'A':0.10  'W':0.07  'I':0.06
after ':'  -> '\n':0.85  ' ':0.15  '-':0.00  "'":0.00  'd':0.00


### 2.4 Generated sample

In [7]:
for prompt, text in results["bigram"]["samples"].items():
    print(f"--- prompt {prompt!r} ---\n{text}\n")

--- prompt '\n' ---

Anid trdly le swen's weangoud th heas tin and my f pomp engenkisod pee'tonoreld tthe wout t ma suchisoswher stotree ier thismay se; fome mo's averve our fis d ar are y fous lk whe oneacang phe.
He, fopanl neens ar ainconensathouch an sthin athis t s band LInld har me f mee hor mit I thanoulour-fo oooue be theached thareas the;
APARD f Morsouer s ade wan d in ke me
Henco t kes?
HARIfonopen s; by meshas herivel he oris she bece belarous be mpre wesen wale o t
Wadse y.



d!
Sheamy ondaveere I thanerd rrfol s. ull,


MELE:
Thewate amathade g bur tr s loure?

We'she, s ngou d hind.
BE rby
Thil!
Pe

--- prompt 'ROMEO:\n' ---
ROMEO:
Anid trdly le swen's weangoud th heas tin and my f pomp engenkisod pee'tonoreld tthe wout t ma suchisoswher stotree ier thismay se; fome mo's averve our fis d ar are y fous lk whe oneacang phe.
He, fopanl neens ar ainconensathouch an sthin athis t s band LInld har me f mee hor mit I thanoulour-fo oooue be theached thareas the;
APARD f Morsoue

The bigram sample is mostly made up of letter groups that are reminiscent of English but do not make words. It reproduces the shape of a speaker tag — a short run of capitals, a colon, then a line break — in S:, O: and IS:, and 7 of the 8 colons in the sample are followed by a newline. What it never produces is an actual name, because a table holding one character of context cannot hold one. There are also stray symbols such as &, which shows up only 3 times in the whole 1.1 million characters — add-1 smoothing is what gives it a floor probability after every character, so the smoothing that makes the model usable also distorts it.

## 3. Model 2 — LSTM

Embedding → 2-layer LSTM → linear head. Hidden size 358 was chosen so the LSTM
has the same number of parameters as the transformer (within 0.03%).

In [8]:
models["lstm"], results["lstm"] = run("lstm", cfg)


== LSTM (1,854,455 parameters) on cuda ==
  step     0 | train 4.1809 | val 4.1813 |    0.0s
  step   250 | train 1.6535 | val 1.7802 |   19.8s
  step   500 | train 1.4023 | val 1.5692 |   39.6s
  step   750 | train 1.3105 | val 1.5033 |   60.6s
  step  1000 | train 1.2554 | val 1.4766 |   83.3s
  step  1250 | train 1.2159 | val 1.4557 |  108.0s
  step  1500 | train 1.1821 | val 1.4404 |  133.5s
  step  1750 | train 1.1516 | val 1.4334 |  158.1s
  step  2000 | train 1.1272 | val 1.4328 |  182.8s
  step  2250 | train 1.1053 | val 1.4310 |  207.6s
  step  2500 | train 1.0817 | val 1.4330 |  232.3s
  step  2750 | train 1.0635 | val 1.4354 |  257.0s
  step  3000 | train 1.0478 | val 1.4353 |  281.7s
  step  3250 | train 1.0297 | val 1.4389 |  306.5s
  step  3500 | train 1.0170 | val 1.4426 |  331.3s
  step  3750 | train 1.0050 | val 1.4448 |  356.0s
  step  4000 | train 0.9952 | val 1.4466 |  380.7s
  step  4250 | train 0.9870 | val 1.4536 |  405.4s
  step  4500 | train 0.9802 | val 1.453

### 3.1 Generated sample — same prompts, seed and temperature as the bigram

In [9]:
for prompt, text in results["lstm"]["samples"].items():
    print(f"--- prompt {prompt!r} ---\n{text}\n")

--- prompt '\n' ---

And a revenge that thou wilt be my affair.

KING RICHARD III:
No, madam, may I think it well.

KING EDWARD IV:
Now, by the son of I, as I respected,
The first lives to quit our love and bear,
Which for my needs are sworn as we he will.

LADY ANNE:
Affairs thee here with envyou to my brother.

EXETER:
The gods come to your grace and make me well.

QUEEN ELIZABETH:

KING RICHARD III:
Well, then you shall ever leave to beg
Your kind home by my father; who has found
For this shame, whose were I kill'd his son,
I sent them to be a courterader.

ESCALUS:
I think you have not made.

CORIOLANUS:
I'll 

--- prompt 'ROMEO:\n' ---
ROMEO:
Ay, ay, and then he was since the fair departure
Let it encounter me: 'tis not the winter
A man between his son.

BRUTUS:
Let him be strike:
I should be first and witnesses of the one,
Who shall not open one so are away.

MENENIUS:
I stay to the man banished; I will give up ourselves
Shall I stow out my tongue.

Second Servingman:
Here comes 

An untrained model should score close to the uniform baseline of 4.1744 nats, because its output is near-random. If it does not, something is wired wrong and it is much cheaper to find out now than after a training run.

## 4. Model 3 — Small Transformer

| Component | Code |
| --- | --- |
| Token embeddings | `TransformerLM.tok_emb` |
| Positional embeddings (learned) | `TransformerLM.pos_emb` |
| Causal multi-head self-attention | `CausalSelfAttention` |
| Feedforward layer | `FeedForward` |
| Output layer | `TransformerLM.ln_f` + `head` |

4 pre-norm blocks, 6 heads, embedding size 192, context 256.

### 4.1 The causal mask is tested, not assumed

If the mask leaked, the model could read the answer, and the loss would look
great while the model learned nothing. The tests change a later character and
check that earlier predictions stay exactly the same.

In [10]:
!{sys.executable} -m pytest -q test_models.py

.....                                                                    [100%]
5 passed in 2.52s


### 4.2 Training — identical loop, batches, schedule and step budget to the LSTM

In [ ]:
models["transformer"], results["transformer"] = run("transformer", cfg)


== Transformer (1,854,017 parameters) on cuda ==
  step     0 | train 4.3569 | val 4.3504 |    0.0s
  step   250 | train 2.3946 | val 2.4188 |   36.6s


### 4.3 Generated sample

In [ ]:
for prompt, text in results["transformer"]["samples"].items():
    print(f"--- prompt {prompt!r} ---\n{text}\n")

The transformer sample includes recognisable character names such as JOHN OF GAUNT, MENENIUS and Grumio, who is an actual character in The Taming of the Shrew. The generated clauses are a little longer and syntactically coherent, although still semantically empty. I found it hard to tell this sample apart from the LSTM's in terms of quality, which is not surprising given the small 0.023-nat difference in validation loss; it would be an exaggeration to claim any noticeable difference.



### 4.4 What the attention heads look at

Attention weights for every head in the first and last block, on the first 64
characters of the validation set. Rows are the position being predicted from,
columns are the earlier positions it reads, and the axes are labelled with the
actual characters, so you can see *what* a head attends to and not just where.

In [ ]:
tf = models["transformer"]
n_show = 64
x = data.val_data[:n_show].unsqueeze(0).to(device)
labels = [c if c not in " \n" else {" ": "␣", "\n": "↵"}[c] for c in decode(x[0].tolist())]

layers = [0, len(tf.blocks) - 1]
for b in tf.blocks:
    b.attn.keep_weights = True
tf.eval()
with torch.no_grad():
    tf(x)

n_head = cfg["n_head"]
fig, axes = plt.subplots(len(layers), n_head, figsize=(3.2 * n_head, 3.4 * len(layers)))
for r, layer in enumerate(layers):
    w = tf.blocks[layer].attn.last_weights[0].cpu()
    for h in range(n_head):
        ax = axes[r, h]
        ax.imshow(w[h], cmap="viridis", vmin=0, vmax=w[h].max())
        ax.set_title(f"block {layer} · head {h}", fontsize=9)
        ax.set_xticks(range(n_show)); ax.set_xticklabels(labels, fontsize=4)
        ax.set_yticks(range(n_show)); ax.set_yticklabels(labels, fontsize=4)
for b in tf.blocks:
    b.attn.keep_weights = False
fig.tight_layout()
os.makedirs("results", exist_ok=True)
fig.savefig("results/attention.png", dpi=160)
plt.show()

above_diag = torch.triu(torch.ones(n_show, n_show, dtype=torch.bool), diagonal=1)
leak = max(b.attn.last_weights[0][:, above_diag].max().item() for b in tf.blocks)
print(f"largest attention weight above the diagonal, any block: {leak:.3e}")
print("text shown:", repr(decode(x[0].tolist())))

**What the heads do.** > ✏️ **Your turn** — write this after reading the output above. Delete this line when done.

## 5. Results

### 5.1 Results table

In [ ]:
from IPython.display import Markdown, display
table = save_results(results, cfg)
display(Markdown(table))
print(f"baselines: uniform {base['uniform']:.4f} | unigram {base['unigram']:.4f} "
      f"| closed-form bigram {counts['val_loss']:.4f}")

Losses are nats per character over **every** character of each split (not a
sample). *Best Val* is the lowest validation loss at any checkpoint, then
re-scored on the full validation set. The *Generated Text Quality* column is a
judgement, so fill it in by hand in `results/results_table.md` and in the README.

### 5.2 Learning curves

In [ ]:
fig = plot_curves(results, "results/learning_curves.png", base)
plt.show()

In [ ]:
print(f"{'step':>6}" + "".join(f"{DISPLAY[n]:>14}" for n in ("lstm", "transformer")))
h_l, h_t = results["lstm"]["history"], results["transformer"]["history"]
for i, step in enumerate(h_l["step"]):
    print(f"{step:>6}{h_l['val'][i]:>14.4f}{h_t['val'][i]:>14.4f}")

### 5.3 Training speed

In [ ]:
for name, r in results.items():
    per_step = 1000 * r["train_time_s"] / cfg["max_steps"]
    print(f"{DISPLAY[name]:12} {r['train_time_s']:7.1f}s total | {per_step:6.2f} ms/step")

### 5.4 Samples side by side (prompt `ROMEO:`)

In [ ]:
for name, r in results.items():
    print("=" * 70)
    print(f"{DISPLAY[name]}  —  val loss {r['final']['val_loss']:.4f}")
    print("=" * 70)
    print(r["samples"]["ROMEO:\n"][:400], "\n")

### 5.5 Qualitative analysis

Which model performed best?

The small transformer had the lowest validation loss at the end of training, 1.5219 nats, against the LSTM's 1.5445 and the bigram's 2.4822. The LSTM, however, reached the lowest validation loss of any single checkpoint — 1.5048 at step 2500, against the transformer's best of 1.5180 at step 4000 — before it began to overfit. So the transformer wins at the fixed 5000-step budget, and the LSTM wins if each model is judged on its best checkpoint. Both of those are worth stating, because which one you report decides the answer.

Which model trained fastest?

The bigram involved no gradient-descent training at all; it is a single counting pass and took 0.0036 seconds, which is a different kind of thing rather than a faster version of the same thing. Of the two trained networks the LSTM was faster, needing 104.7 seconds for 5000 steps against the transformer's 166.7 seconds for the same number of steps. The transformer is more expensive per step because every one of its 4 blocks computes self-attention over the whole 128-character window and then a feedforward layer at every position, which illustrates that being parallelisable does not automatically mean faster wall-clock time.

One limitation of these two numbers. They are wall-clock times taken with time.perf_counter() around the training step, and reading the per-500-step figures back out shows the two models' timings behaving differently: the LSTM's chunks rise steadily from 9.7s to 11.6s across the run, a 19.6% increase on a workload that does not change, while the transformer's stay flat between 16.3s and 16.8s. I did not establish what causes that difference, so the ratio between the two totals should be read as what this run measured rather than as a fixed property of the two architectures.

Which model generated the most realistic text?

The bigram was clearly last: it produces letter clusters that look vaguely English rather than words, and the speaker tags it produces (S:, O:, IS:) have the right punctuation shape but are not names. The LSTM and the transformer both produced realistic Shakespearean dialogue structure, with correct NAME: tags and real character names from the corpus — FLORIZEL:, DUCHESS OF YORK: and AUFIDIUS: from the LSTM, JOHN OF GAUNT:, MENENIUS: and SICINIUS: from the transformer. Between those two I could not reliably tell which was better. That matches the small 0.023-nat difference in validation loss — a gap that size should not be visible by eye, and claiming to see one would be reading more into the samples than they support.

What did the transformer improve compared to the other models?

Against the bigram it improved almost everything — spelling, word shapes, real speaker names rather than just the punctuation shape of a tag, and dependencies reaching much further back than one character (2.4822 down to 1.5219 nats, or a perplexity of 11.97 down to 4.58). Against the LSTM the gain in final validation loss was small, 1.5219 against 1.5445, and the more interesting difference is in how the two models behaved after their best checkpoint.

Both of them peaked and then got worse, but by very different amounts. The LSTM peaked at step 2500 at 1.5048 and finished at 1.5445, giving back 0.0396 nats. The transformer peaked later, at step 4000 at 1.5180, and finished at 1.5219, giving back 0.0039 — about ten times less. The LSTM also drove its training loss 0.104 lower than the transformer managed (1.0792 against 1.1834) while ending with the worse validation loss, which is overfitting visible directly in the two numbers. The train/validation gap says the same thing: 0.465 for the LSTM against 0.339 for the transformer.

One caveat before reading that gap as a property of the architectures. At the same nominal dropout of 0.1, the transformer carries eight dropout sites — one in the attention and one in the feedforward layer of each of its four blocks — while the LSTM carries one, because nn.LSTM applies dropout only between stacked layers. The transformer is the more heavily regularised of the two models, and that alone is enough to shrink its train/validation gap, so the gap is not evidence about attention on its own.

Was the improvement worth the added complexity?

In this limited setting — 1.1 million characters, and 841,905 parameters in the LSTM against 826,433 in the transformer — the difference in accuracy between the two is marginal, the transformer being ahead by only 0.023 nats in final validation loss. Since it takes around 60% longer to train (166.7s against 104.7s) and a good deal more code, that overhead does not look worth it at this problem size.

What stops me generalising from that is how differently the two models held up past their best checkpoint. Over the last 2500 steps the LSTM gave back 0.0396 nats and the transformer 0.0039, and that difference is the whole reason the transformer ends ahead on the final number despite having the worse best checkpoint. It is the one result here that bears on what might happen with a longer run, more data, or a longer context — though with a 5000-step budget on a 1.1 MB corpus I can only say that this run points that way, not that it demonstrates it.

## 6. Reflection

What I learned about attention
The heatmap of block 0 shows that the upper triangle is completely empty in all four heads, which is the causal mask doing its job — no head can see the character the model is being asked to predict. The check in §4.4 agrees: the largest attention weight above the diagonal is exactly 0.000e+00. This is worth testing rather than assuming, because a broken mask fails silently, with a loss that looks excellent and a model that has learned nothing.

What I did not expect is that the four heads do not behave the same way. Heads 1 and 2 are almost entirely local — a single bright diagonal, meaning each position puts nearly all of its weight on the character immediately at hand. Heads 0 and 3 spread their weight much more widely over earlier positions, and head 3 shows a bright vertical stripe where many later positions attend back to one particular earlier character. Nothing in the loss function asks the heads to divide the work up this way; they settle into different jobs on their own. This is one attention map from one validation sequence, so it is an illustration of what the trained model does here rather than a measurement across the whole validation set.

Final reflection
After comparing bigram, non-transformer, and transformer models, what makes transformers powerful for language modelling?

At this size — around a million characters and under a million parameters — there was very little between the LSTM and the transformer, which says more about how strong a well-tuned recurrent network still is than about either architecture winning. It is worth being precise about why: both models were given the same 128-character window, so the transformer's advantage here was never that it could see further back.

What makes transformers powerful is how they scale, rather than anything dramatic at this size. Self-attention removes the sequential bottleneck, so the positions in a window can be computed in parallel instead of one after another, and every position can refer directly to any earlier one without that information having to survive being compressed into a single running state. My own run only hints at this rather than demonstrating it, and the hint is in what happened after each model's best checkpoint: both of them peaked before the budget ran out — the LSTM at step 2500 and the transformer at step 4000 — but over the remaining steps the LSTM gave back 0.0396 nats while the transformer gave back 0.0039. The parallelism did not even pay for itself in wall-clock time here — the transformer was 60% slower — which is a reminder that these advantages are claims about what happens when the model, the data and the context all get much larger, not about a 1.1 MB corpus.